## 1. Import Libraries

We import PyTorch (`torch`) instead of `tensorflow`. We'll also use plain Python (`collections.Counter`) to build our own vocabulary, since PyTorch doesn't ship a `Tokenizer` class the way Keras does.

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from collections import Counter
import pickle

# This lets the notebook automatically use a GPU if one is available,
# and fall back to the CPU otherwise. Training on GPU is much faster.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


## 2. Load and Explore the Dataset

Nothing changes here — loading a CSV with `pandas` works the same way regardless of which deep learning library you use later.

In [4]:
df = pd.read_csv(r"C:\Users\rohit\Downloads\AI and Data Science\𝗗𝗲𝗲𝗽 𝗟𝗲𝗮𝗿𝗻𝗶𝗻𝗴 𝗣𝗿𝗼𝗷𝗲𝗰𝘁𝘀\RNN Projects\Sentence Auto Completer Bot\data\qoute_dataset.csv")


In [5]:
df.head()


,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [7]:
df['quote'][89]

'“I believe in Christianity as I believe that the sun has risen: not only because I see it, but because by it I see everything else.”'

In [8]:
df.shape


(3038, 2)

In [9]:
quotes = df['quote']
quotes.head()


0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

## 3. Clean the Text

We lowercase every quote and strip punctuation, exactly like in the original notebook. This keeps the vocabulary small and consistent (so "Hello" and "hello," are treated as the same word).

In [10]:
quotes = quotes.str.lower()


In [11]:
import string

translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))


In [12]:
quotes.head()


0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

## 4. Build a Vocabulary (PyTorch has no `Tokenizer`)

Keras gives you a ready-made `Tokenizer` class that counts words and assigns each one a number. PyTorch does not include an equivalent, so we build one ourselves — it's only a few lines:

1. Count how often every word appears across all quotes, using `Counter`.
2. Keep only the `vocab_size - 1` most common words (we reserve **index 0** for padding — more on padding in Section 6).
3. Create two lookup dictionaries:
   - `word_to_index`: word → number (used to turn text into numbers)
   - `index_to_word`: number → word (used later to turn predictions back into words)

Any word that isn't frequent enough to make it into the vocabulary is simply skipped when we convert text to numbers, the same way Keras's `Tokenizer(num_words=...)` behaves.

In [13]:
vocab_size = 10000

def build_vocab(text_series, vocab_size):
    counter = Counter()
    for quote in text_series:
        counter.update(quote.split())

    # Reserve index 0 for padding, so the vocabulary itself only
    # gets indices 1 .. vocab_size - 1
    most_common_words = counter.most_common(vocab_size - 1)

    word_to_index = {word: idx + 1 for idx, (word, _count) in enumerate(most_common_words)}
    index_to_word = {idx: word for word, idx in word_to_index.items()}
    return word_to_index, index_to_word

word_to_index, index_to_word = build_vocab(quotes, vocab_size)

print("Vocabulary size (including padding index 0):", len(word_to_index) + 1)
list(word_to_index.items())[:10]


Vocabulary size (including padding index 0): 8979


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

## 5. Convert Quotes into Sequences of Numbers

This does the same job as `tokenizer.texts_to_sequences(...)` in Keras: turn each quote (a string) into a list of integers, using the `word_to_index` dictionary we just built. Words that didn't make it into the vocabulary are simply dropped.

In [14]:
def text_to_sequence(text, word_to_index):
    return [word_to_index[word] for word in text.split() if word in word_to_index]

sequences = [text_to_sequence(quote, word_to_index) for quote in quotes]


In [15]:
for i in range(3):
    print(quotes.iloc[i])


“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [16]:
for i in range(3):
    print(sequences[i])


[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


## 6. Create Input → Next-Word Pairs

This is the core idea of "next-word prediction" as a training task, and it doesn't change between frameworks.

For every quote, we slide through it and create training examples like:

```
"the world is beautiful"  ->  [the]              -> world
                              [the, world]        -> is
                              [the, world, is]    -> beautiful
```

So `X` holds the growing list of "words so far", and `y` holds the single word that came next.

In [17]:
X = []
y = []

for seq in sequences:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)


In [18]:
len(X)


85271

In [19]:
len(y)


85271

In [20]:
max_len = max(len(x) for x in X)
print(max_len)


745


## 7. Pad the Sequences

Every example in `X` has a different length (some quotes are 2 words in, others are 20 words in), but a neural network needs every input in a batch to be the same size. Keras solved this with `pad_sequences(..., padding='pre')`, which adds zeros to the **front** of short sequences.

PyTorch has no built-in equivalent for this exact behaviour, so we write a small NumPy function that does the same thing: pad with zeros on the left so every sequence ends up with length `max_len`.

We use **pre-padding** (zeros at the start) to match the original notebook — this also matches how index `0` was reserved for padding back in Section 4.

In [21]:
def pad_sequences_pre(sequences, max_len):
    padded = np.zeros((len(sequences), max_len), dtype=np.int64)
    for i, seq in enumerate(sequences):
        length = min(len(seq), max_len)
        # place the (truncated) sequence at the END of the row,
        # leaving zeros at the start -> this is "pre" padding
        padded[i, -length:] = seq[-length:]
    return padded

X_padded = pad_sequences_pre(X, max_len)


In [22]:
y = np.array(y)


In [23]:
X_padded.shape


(85271, 745)

In [24]:
y.shape


(85271,)

## 8. Skip One-Hot Encoding — PyTorch Doesn't Need It

The original notebook used `to_categorical(y, num_classes=vocab_size)` to turn each target word into a one-hot vector, because Keras's `categorical_crossentropy` loss expects that format.

**In PyTorch, we don't do this.** PyTorch's `nn.CrossEntropyLoss` expects the target as a plain class index (an integer, like `y` already is) — it applies softmax and does the one-hot comparison internally, in a numerically more stable way. So this step is simply deleted, and `y` is used exactly as it is.

## 9. Build a PyTorch `Dataset` and `DataLoader`

Keras just takes `X` and `y` arrays directly into `model.fit(...)`. PyTorch instead asks you to wrap your data in a `Dataset` class, so it knows how to fetch one example at a time, and a `DataLoader`, which handles batching, shuffling, and iterating for you during training.

In [25]:
class QuoteDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64

dataset = QuoteDataset(X_padded, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


## 10. Define the RNN Model

In Keras, `Sequential()` + `.add(...)` builds the model for you layer by layer. In PyTorch, you write a small class that inherits from `nn.Module`, with:

- `__init__`: define the layers you'll use (same layers as the Keras version: `Embedding` → `RNN` → `Dense`/`Linear`)
- `forward`: define how data actually flows through those layers

This is more explicit than Keras, but it also means there's no hidden "magic" — you can see exactly what happens to the input at every step.

Note: we only need the RNN's output at the **last timestep**, since that's the word right after everything we've seen so far — this plays the same role as Keras's `SimpleRNN(units=...)` returning only its final output by default.

In [26]:
embedding_dim = 50
rnn_units = 128

class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        # padding_idx=0 tells PyTorch that index 0 is "just padding",
        # so it won't waste effort learning a meaningful vector for it
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=rnn_units, batch_first=True)
        self.fc = nn.Linear(rnn_units, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)              # (batch, seq_len, embedding_dim)
        rnn_output, _hidden = self.rnn(embedded)   # rnn_output: (batch, seq_len, rnn_units)
        last_step = rnn_output[:, -1, :]           # keep only the last timestep, like Keras SimpleRNN default
        logits = self.fc(last_step)                # (batch, vocab_size) -- raw scores, no softmax here
        return logits

rnn_model = RNNModel(vocab_size, embedding_dim, rnn_units).to(device)
print(rnn_model)


RNNModel(
  (embedding): Embedding(10000, 50, padding_idx=0)
  (rnn): RNN(50, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=10000, bias=True)
)


### Why no `.compile(...)` step?

Keras's `model.compile(optimizer=..., loss=..., metrics=...)` just stores these settings so `model.fit()` can use them later. In PyTorch we don't "attach" them to the model — we create the loss function and optimizer as separate objects and use them directly in our own training loop (Section 12). This is what Section 8 already hinted at: PyTorch is more manual, but nothing is hidden.

In [27]:
# Quick parameter count, similar in spirit to model.summary() in Keras
total_params = sum(p.numel() for p in rnn_model.parameters())
trainable_params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Total parameters: 1,813,040
Trainable parameters: 1,813,040


## 11. Define the LSTM Model

Same structure as the RNN model above, but swapping `nn.RNN` for `nn.LSTM` — just like the original notebook swapped `SimpleRNN` for `LSTM`. LSTMs are generally better at remembering longer sequences than a plain RNN.

In [28]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=rnn_units, batch_first=True)
        self.fc = nn.Linear(rnn_units, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_output, _hidden = self.lstm(embedded)
        last_step = lstm_output[:, -1, :]
        logits = self.fc(last_step)
        return logits

lstm_model = LSTMModel(vocab_size, embedding_dim, rnn_units).to(device)
print(lstm_model)


LSTMModel(
  (embedding): Embedding(10000, 50, padding_idx=0)
  (lstm): LSTM(50, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=10000, bias=True)
)


In [29]:
total_params = sum(p.numel() for p in lstm_model.parameters())
trainable_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Total parameters: 1,882,160
Trainable parameters: 1,882,160


## 12. Write the Training Loop

This is the biggest difference from Keras. `model.fit(X, y, epochs=...)` does all of this for you in one line. In PyTorch, we write the loop ourselves, so let's break down what it actually does, one epoch (one full pass over the data) at a time:

1. Loop over batches from the `DataLoader`.
2. Move the batch to the `device` (CPU or GPU).
3. Reset gradients from the previous step (`optimizer.zero_grad()`).
4. Run the batch through the model to get predictions (`model(X_batch)`).
5. Compare predictions to the true next word using the loss function.
6. Compute gradients (`loss.backward()`).
7. Update the model's weights (`optimizer.step()`).

We wrap all of this in a reusable function so we can train both the RNN and the LSTM the same way.

In [30]:
def train_model(model, dataloader, epochs, learning_rate=0.001):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

        avg_loss = total_loss / total
        accuracy = correct / total
        print(f"Epoch {epoch+1}/{epochs} - loss: {avg_loss:.4f} - accuracy: {accuracy:.4f}")

    return model


## 13. Train the RNN Model

A note before you run this: with `vocab_size = 10000`, training can be slow on a CPU. Feel free to lower `epochs` (or `vocab_size`, back in Section 4) while you're first testing that everything runs correctly, then increase it once you're ready for a longer training run.

In [ ]:
epochs = 5

rnn_model = train_model(rnn_model, dataloader, epochs=epochs)


## 14. Train the LSTM Model

Same function, different model — this is the benefit of writing `train_model` as a reusable function instead of repeating the loop.

In [ ]:
lstm_model = train_model(lstm_model, dataloader, epochs=epochs)


## 15. Save and Load the Model

Keras saves the whole model (architecture + weights) into a single `.h5` file. PyTorch's standard practice is a little different: you save only the learned weights (the `state_dict`), and keep the model's class definition (Sections 10/11) in your code. To reload the model later, you re-create the class and load the saved weights into it.

In [ ]:
torch.save(lstm_model.state_dict(), "lstm_model.pth")


In [ ]:
# Re-creating the model and loading the saved weights back in.
# You need the LSTMModel class (Section 11) available to do this.
loaded_lstm_model = LSTMModel(vocab_size, embedding_dim, rnn_units)
loaded_lstm_model.load_state_dict(torch.load("lstm_model.pth", map_location=device))
loaded_lstm_model.to(device)
loaded_lstm_model.eval()   # switch to evaluation mode -- turns off training-only behaviour

lstm_model = loaded_lstm_model


## 16. Predict the Next Word

This mirrors the original `predictor(...)` function closely:

1. Lowercase the input text.
2. Convert it to a sequence of numbers using our vocabulary.
3. Pad it to `max_len`.
4. Run it through the model and take the word with the highest score.

Two PyTorch-specific details worth calling out:
- `model.eval()` tells the model we're doing inference, not training (this matters for some layer types, and is good practice generally).
- `torch.no_grad()` tells PyTorch not to bother tracking gradients here, since we're not training — this makes prediction faster and uses less memory.

In [ ]:
def predictor(model, word_to_index, index_to_word, text, max_len):
    model.eval()
    text = text.lower()

    seq = text_to_sequence(text, word_to_index)
    padded = pad_sequences_pre([seq], max_len)
    input_tensor = torch.tensor(padded, dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        pred_index = torch.argmax(logits, dim=1).item()

    return index_to_word.get(pred_index, "")


In [ ]:
seed_text = "what are you"
next_word = predictor(lstm_model, word_to_index, index_to_word, seed_text, max_len)
print(next_word)


## 17. Generate a Longer Sequence of Text

Same idea as before: repeatedly predict one word, add it to the sentence, and feed the (now longer) sentence back in for the next prediction.

In [ ]:
def generate_text(model, word_to_index, index_to_word, seed_text, max_len, n_words):
    for _ in range(n_words):
        next_word = predictor(model, word_to_index, index_to_word, seed_text, max_len)
        if next_word == "":
            break
        seed_text += " " + next_word
    return seed_text


In [ ]:
seed = "are you a"
generated = generate_text(lstm_model, word_to_index, index_to_word, seed, max_len, 10)
print(generated)


## 18. Save the Vocabulary and `max_len`

If you want to reuse this trained model somewhere else (e.g. a small script or app), you need to save `word_to_index`, `index_to_word`, and `max_len` too — not just the model weights — since they're required to turn new text into numbers the model understands, and to turn its predictions back into words.

In [ ]:
with open("vocab.pkl", "wb") as f:
    pickle.dump({"word_to_index": word_to_index, "index_to_word": index_to_word}, f)

with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)
